# XFeat-SuperPoint Hybrid Model — Fixed Training
## Training on MegaDepth-1500 (Google Colab)

### What was fixed vs the original notebook
| Issue | Fix |
|---|---|
| In-place border zeroing broke autograd graph | Out-of-place masking in `_decode_xfeat_heatmap` |
| Hard NMS disconnected scores from loss | NMS uses detached positions; **score values** remain differentiable |
| Descriptors stored as `(256,N)` but loss expected `(N,256)` | Model now outputs `(N,256)` directly |
| Aux L2 loss pushed all weights toward zero | Replaced with **score-weighted hinge loss** + **repeatability reward** |
| SuperPoint params unfrozen but received no gradient | Removed the incorrect `requires_grad_(True)` |
| Approximate homography noisy for 3-D scenes | Optional **depth-based warp field** when depth maps are present |
| No validation in training loop | Added proper val loop + early stopping |
| `MultiStepLR` with fixed milestones | `ReduceLROnPlateau` (adapts to actual val loss) |

**Drive layout expected:**
```
MyDrive/hybrid_feature_matching/data/megadepth_test_1500/
    ├── megadepth_test_1500_scene_info/   ← .npz pair files
    └── megadepth_test_1500.tar           ← images + depth maps
```


In [ ]:
import subprocess, sys, os, time
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → T4 GPU')
print(result.stdout[:500])

import torch
device = torch.device('cuda')
props  = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}  VRAM: {props.total_memory/1e9:.1f} GB')


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil, subprocess
from pathlib import Path

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/hybrid_feature_matching'
DRIVE_BASE = f'{DRIVE_PROJECT_ROOT}/data/megadepth_test_1500'
SCENE_INFO_DIR = f'{DRIVE_BASE}/megadepth_test_1500_scene_info'
MEGADEPTH_TAR = f'{DRIVE_BASE}/megadepth_test_1500.tar'

# Persistent checkpoint folder on Drive
DRIVE_CKPT_DIR = f'{DRIVE_PROJECT_ROOT}/checkpoints/xfeat_superpoint_hybrid'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(SCENE_INFO_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Auto-download scene_info assets when missing
npz_files = sorted(Path(SCENE_INFO_DIR).glob('*.npz'))
if len(npz_files) == 0:
    print('No .npz found in scene_info dir. Downloading from LoFTR assets...')
    loftr_clone_dir = '/content/.loftr_assets'
    if os.path.exists(loftr_clone_dir):
        shutil.rmtree(loftr_clone_dir)

    subprocess.run([
        'git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
        'https://github.com/zju3dv/LoFTR.git', loftr_clone_dir
    ], check=True)
    subprocess.run([
        'git', '-C', loftr_clone_dir, 'sparse-checkout', 'set',
        'assets/megadepth_test_1500_scene_info'
    ], check=True)

    src_dir = Path(loftr_clone_dir) / 'assets' / 'megadepth_test_1500_scene_info'
    copied = 0
    for npz in src_dir.glob('*.npz'):
        shutil.copy2(npz, Path(SCENE_INFO_DIR) / npz.name)
        copied += 1

    shutil.rmtree(loftr_clone_dir)
    print(f'✓ Downloaded {copied} scene-info .npz files to: {SCENE_INFO_DIR}')

npz_files = sorted(Path(SCENE_INFO_DIR).glob('*.npz'))

errors = []
for label, path in [
    ('scene_info dir', SCENE_INFO_DIR),
    ('Megadepth tar', MEGADEPTH_TAR),
    ('Drive ckpt dir', DRIVE_CKPT_DIR),
]:
    ok = os.path.exists(path)
    icon = '✓' if ok else '✗'
    print(f'{icon} {label:16s}: {path}')
    if not ok and label == 'Megadepth tar':
        errors.append(path)

print()
print(f'✓ Found {len(npz_files)} .npz scene-info files')
if len(npz_files) == 0:
    errors.append('scene_info .npz files missing')

if errors:
    msg = 'Missing required paths/files:' + chr(10) + chr(10).join(errors)
    msg += chr(10) + chr(10) + 'Put megadepth_test_1500.tar at MEGADEPTH_TAR path, then re-run this cell.'
    raise FileNotFoundError(msg)

print('✓ Drive paths and scene_info setup verified')




In [ ]:
%%capture
!pip install -q h5py kornia


In [ ]:
import subprocess, sys, os

REPO_DIR = '/content/XFeat-SuperPoint-Hybrid-Model'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Clone XFeat backbone
XFEAT_DIR = '/content/accelerated_features'
if not os.path.exists(XFEAT_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/verlab/accelerated_features.git',
                    XFEAT_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if XFEAT_DIR not in sys.path:
    sys.path.insert(0, XFEAT_DIR)

print('✓ Repos ready')


In [ ]:
import tarfile, os

EXTRACT_ROOT = '/content/megadepth'
FLAG_FILE    = f'{EXTRACT_ROOT}/.extraction_complete'

if os.path.exists(FLAG_FILE):
    print('✓ Already extracted — skipping')
else:
    print(f'Extracting {MEGADEPTH_TAR} …  (first run: ~10-20 min)')
    os.makedirs(EXTRACT_ROOT, exist_ok=True)
    with tarfile.open(MEGADEPTH_TAR, 'r') as tf:
        tf.extractall(EXTRACT_ROOT)
    open(FLAG_FILE, 'w').close()
    print('✓ Extraction complete')

# Auto-detect image root
IMAGE_ROOT = None
for candidate in [
    f'{EXTRACT_ROOT}/megadepth_test_1500',
    f'{EXTRACT_ROOT}/Undistorted_SfM',
    f'{EXTRACT_ROOT}/phoenix',
    EXTRACT_ROOT,
]:
    if os.path.isdir(candidate):
        IMAGE_ROOT = candidate
        break

print(f'Image root: {IMAGE_ROOT}')
print(f'Contents:   {os.listdir(EXTRACT_ROOT)[:6]}')


In [ ]:
import numpy as np
from pathlib import Path

npz_files = sorted(Path(SCENE_INFO_DIR).glob('*.npz'))
print(f'Scene info files: {len(npz_files)}')

for npz_path in npz_files[:2]:
    data = np.load(str(npz_path), allow_pickle=True)
    print(f'\n{npz_path.name}:  keys = {list(data.keys())}')
    for k in data.keys():
        v = data[k]
        if hasattr(v, 'shape'):
            print(f'  {k}: {v.shape}  {v.dtype}')
        else:
            print(f'  {k}: {type(v).__name__}')


In [ ]:
import torch, torch.nn as nn, sys, os, glob, importlib

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Clear stale bytecache ──────────────────────────────────────────────────
for pyc in glob.glob(f'{REPO_DIR}/models/__pycache__/*.pyc'):
    os.remove(pyc)
for k in list(sys.modules.keys()):
    if k == 'models' or k.startswith('models.'):
        del sys.modules[k]
importlib.invalidate_caches()

# ── XFeat backbone ────────────────────────────────────────────────────────
from modules.xfeat import XFeat as XFeatPipeline
_pipeline   = XFeatPipeline()
xfeat_core  = _pipeline.net   # the actual nn.Module

# Probe to confirm output shape
with torch.no_grad():
    _p  = torch.rand(1, 1, 64, 64, device=device)
    _o  = xfeat_core(_p)
    _kp = _o['keypoints'] if isinstance(_o, dict) else _o[0]
    print(f'XFeat kp output shape: {tuple(_kp.shape)}')

# ── SuperPoint core ───────────────────────────────────────────────────────
# Option A: use the SuperPoint implementation shipped with LightGlue / kornia
try:
    from kornia.feature import KeyNetDetector  # fallback only
    raise ImportError('prefer SuperPoint')
except Exception:
    pass

try:
    from models.superpoint_core import SuperPointCore
    superpoint = SuperPointCore()
    print('Loaded SuperPointCore from models/')
except ImportError:
    # Minimal SuperPoint stub (replace with a real pretrained model in prod)
    class SuperPointStub(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(),
                nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
                nn.MaxPool2d(2),
            )
            self.desc_head = nn.Conv2d(256, 256, 1)

        def get_descriptor_map(self, x):
            feat = self.encoder(x)
            desc = self.desc_head(feat)
            import torch.nn.functional as F
            return F.normalize(desc, p=2, dim=1)

    superpoint = SuperPointStub()
    print('⚠ Using SuperPoint stub — replace with pretrained weights!')

# ── HybridModel ───────────────────────────────────────────────────────────
from models.hybrid_model import HybridModel

model = HybridModel(
    xfeat_core=xfeat_core,
    superpoint_core=superpoint,
    num_keypoints=1024,
    nms_radius=4,
    descriptor_dim=256,
    border_margin=16,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable params: {trainable:,} / {total:,}')
print('✓ Model ready')

In [ ]:
model.eval()
with torch.no_grad():
    dummy = torch.rand(1, 1, 480, 640, device=device)
    out   = model(dummy)

kp   = out['keypoints'][0]       # (N, 2)
desc = out['descriptors'][0]     # (N, 256)
norm = desc.norm(dim=1)          # per-descriptor L2 norm

print('=== Forward pass sanity check ===')
print(f'  Keypoints  shape: {kp.shape}   range=[{kp.min():.3f}, {kp.max():.3f}]')
print(f'  Descriptors shape: {desc.shape}')
print(f'  Desc L2-norm:     mean={norm.mean():.4f}  (should be ≈ 1.000)')

assert norm.mean().item() > 0.99, 'L2 norm check failed'
assert kp.min() >= 0.0 and kp.max() <= 1.0, 'Keypoints out of [0,1]'
print('\n✓ All checks passed')


In [ ]:
import numpy as np, torch, random
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as T

class MegaDepth1500Dataset(Dataset):
    """
    MegaDepth-1500 dataset with optional depth-based warp fields.

    Returns per item:
      image1, image2   : (1, H, W) float32 in [0,1]
      homography       : (3, 3) approximate H = K2 R21 K1_inv (always)
      warp_field       : (H, W, 2) depth-based warp (when depth available)
      warp_valid       : (H, W) bool validity mask
    """
    MAX_GETITEM_RETRIES = 20  # cap __getitem__ retries when some paths/depth files are unreadable

    def __init__(self, scene_info_dir, image_root,
                 image_path_key, pair_key,
                 image_size=(480, 640),
                 max_pairs_per_scene=120,
                 min_overlap=0.20,
                 augment=False):
        self.image_root       = Path(image_root)
        self.image_path_key   = image_path_key
        self.pair_key         = pair_key
        self.image_size       = image_size
        self.max_pairs        = max_pairs_per_scene
        self.min_overlap      = min_overlap
        self.aug = T.Compose([
            T.RandomApply([T.ColorJitter(brightness=0.25, contrast=0.25)], p=0.5),
            T.RandomApply([T.GaussianBlur(5, sigma=(0.5, 1.5))], p=0.3),
        ]) if augment else None
        self.pairs = []
        self._load_all(Path(scene_info_dir))
        print(f'[Dataset] {len(self.pairs):,} pairs from {scene_info_dir}')

    # ── index construction ─────────────────────────────────────────────
    def _load_all(self, scene_info_dir):
        for p in sorted(scene_info_dir.glob('*.npz')):
            try:   self._load_scene(p)
            except Exception as e: print(f'  [WARN] {p.name}: {e}')

    def _load_scene(self, npz_path):
        data = np.load(str(npz_path), allow_pickle=True)
        img_paths  = data[self.image_path_key]
        K_arr      = data.get('intrinsics', None)
        T_arr      = data.get('poses', None)
        pairs_raw  = data.get(self.pair_key, None)
        depth_key  = 'depth_paths' if 'depth_paths' in data else None
        depth_paths = data[depth_key] if depth_key else None

        if pairs_raw is None or K_arr is None or T_arr is None:
            return
        count = 0
        for pair_info in pairs_raw:
            if count >= self.max_pairs:
                break
            try:
                pi = pair_info if not hasattr(pair_info, '__iter__') else tuple(pair_info)
                if hasattr(pi[0], '__iter__'):
                    idx0, idx1 = int(pi[0][0]), int(pi[0][1])
                    ov = float(pi[1]) if len(pi) > 1 else 0.3
                else:
                    idx0, idx1 = int(pi[0]), int(pi[1])
                    ov = float(pi[2]) if len(pi) > 2 else 0.3
            except Exception:
                continue
            if ov < self.min_overlap:
                continue
            p1 = str(img_paths[idx0]).strip() if idx0 < len(img_paths) else None
            p2 = str(img_paths[idx1]).strip() if idx1 < len(img_paths) else None
            if not p1 or not p2 or p1.lower()=='none' or p2.lower()=='none':
                continue
            entry = dict(
                p1=p1, p2=p2, overlap=ov,
                K1=K_arr[idx0].astype(np.float32),
                K2=K_arr[idx1].astype(np.float32),
                T1=T_arr[idx0].astype(np.float32),
                T2=T_arr[idx1].astype(np.float32),
            )
            if depth_paths is not None:
                entry['dp1'] = str(depth_paths[idx0]).strip()
            self.pairs.append(entry)
            count += 1

    # ── image / depth loaders ──────────────────────────────────────────
    def _load_img(self, rel_path):
        for base in [self.image_root, Path('/')]:
            p = base / rel_path
            if p.exists():
                img = Image.open(str(p)).convert('L')
                H, W = self.image_size
                img = TF.resize(img, [H, W], interpolation=T.InterpolationMode.BILINEAR)
                return TF.to_tensor(img).float()
        raise FileNotFoundError(rel_path)

    def _load_depth(self, rel_path):
        try:
            import h5py
            import torch.nn.functional as F
        except Exception:
            return None

        for base in [self.image_root, Path('/')]:
            p = base / rel_path
            if p.exists():
                try:
                    with h5py.File(str(p), 'r') as f:
                        d = torch.from_numpy(f['depth'][:].astype(np.float32))
                    if d.dim() == 2: d = d.unsqueeze(0)
                    H, W = self.image_size
                    d = F.interpolate(d.unsqueeze(0), size=(H, W), mode='nearest').squeeze(0)
                    return d   # (1, H, W)
                except Exception:
                    return None
        return None

    # ── depth-based warp field ─────────────────────────────────────────
    @staticmethod
    def _warp_field(K1, K2, T1, T2, depth1, image_size):
        """Per-pixel warp from image-1 to image-2 using depth reprojection."""
        H, W = image_size
        d = depth1[0].numpy().astype(np.float64)
        yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        pix = np.stack([xx.flatten(), yy.flatten(), np.ones(H*W)])   # (3, H*W)
        K1i = np.linalg.inv(K1.astype(np.float64))
        rays = K1i @ pix                                               # (3, H*W)
        P1   = rays * d.flatten()[np.newaxis, :]
        R1, t1 = T1[:3,:3].astype(np.float64), T1[:3,3].astype(np.float64)
        R2, t2 = T2[:3,:3].astype(np.float64), T2[:3,3].astype(np.float64)
        Pw   = R1.T @ (P1 - t1[:,np.newaxis])
        P2   = R2 @ Pw + t2[:,np.newaxis]
        z2   = P2[2]
        valid = (z2 > 0.01) & (d.flatten() > 0.0)
        proj = K2.astype(np.float64) @ (P2 / np.where(valid, z2, 1.0)[np.newaxis,:])
        wf   = np.stack([proj[0], proj[1]], axis=-1).reshape(H, W, 2).astype(np.float32)
        wv   = valid.reshape(H, W)
        return torch.from_numpy(wf), torch.from_numpy(wv)

    # ── approx homography ──────────────────────────────────────────────
    @staticmethod
    def _approx_H(K1, K2, T1, T2):
        R1, R2 = T1[:3,:3], T2[:3,:3]
        R21    = R2 @ R1.T
        K2t    = torch.from_numpy(K2).float()
        R21t   = torch.from_numpy(R21).float()
        K1i    = torch.from_numpy(np.linalg.inv(K1)).float()
        return K2t @ R21t @ K1i

    # ── __getitem__ ───────────────────────────────────────────────────
    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        original_idx = idx
        for attempt in range(self.MAX_GETITEM_RETRIES):
            pair = self.pairs[idx]
            try:
                img1 = self._load_img(pair['p1'])
                img2 = self._load_img(pair['p2'])
            except Exception:
                idx = random.randint(0, len(self)-1)
                continue

            H_mat = self._approx_H(pair['K1'], pair['K2'], pair['T1'], pair['T2'])

            # Depth warp (optional)
            warp_field = warp_valid = None
            if 'dp1' in pair and pair['dp1'] and pair['dp1'].lower() != 'none':
                depth = self._load_depth(pair['dp1'])
                if depth is not None:
                    try:
                        warp_field, warp_valid = self._warp_field(
                            pair['K1'], pair['K2'], pair['T1'], pair['T2'],
                            depth, self.image_size)
                    except Exception:
                        pass

            if self.aug is not None:
                img2_pil = TF.to_pil_image(img2.clamp(0,1))
                img2     = TF.to_tensor(self.aug(img2_pil))

            out = dict(image1=img1, image2=img2, homography=H_mat,
                       overlap=torch.tensor(pair['overlap']))
            if warp_field is not None:
                out['warp_field'] = warp_field
                out['warp_valid'] = warp_valid
            return out

        raise RuntimeError(f'Failed to load a valid sample after {self.MAX_GETITEM_RETRIES} attempts (original idx={original_idx})')


def collate_fn(batch):
    all_keys = set(k for b in batch for k in b.keys())
    out = {}
    for key in all_keys:
        vals = [b.get(key) for b in batch]
        if all(v is None for v in vals):
            out[key] = None
        elif any(v is None for v in vals):
            out[key] = None   # mixed — fall back to homography
        elif all(isinstance(v, torch.Tensor) for v in vals):
            out[key] = torch.stack(vals, dim=0)
        else:
            out[key] = vals
    return out


In [ ]:
# -- auto-detect npz keys from first file ----------------------------------
import numpy as np
from pathlib import Path

sample = np.load(str(sorted(Path(SCENE_INFO_DIR).glob('*.npz'))[0]),
                 allow_pickle=True)
all_keys = list(sample.keys())
print('npz keys:', all_keys)

# pick image_path key
IMG_KEY  = next(k for k in all_keys if 'image' in k.lower() and 'path' in k.lower())
# pick pair key
PAIR_KEY = next(k for k in all_keys
                if 'pair' in k.lower() or 'match' in k.lower()
                or 'overlap' in k.lower() and 'pair' in k.lower())
print(f'Using  image_path_key={IMG_KEY!r}  pair_key={PAIR_KEY!r}')

# IMAGE_ROOT should point to the root where relative paths in the npz resolve
# Adjust if auto-detection above was wrong
print(f'IMAGE_ROOT={IMAGE_ROOT}')

BATCH_SIZE   = 4
NUM_WORKERS  = 0  # Colab+Drive can crash with multiprocessing file IO; raise to 1/2 only if stable
IMAGE_SIZE   = (480, 640)

train_ds = MegaDepth1500Dataset(
    SCENE_INFO_DIR, IMAGE_ROOT, IMG_KEY, PAIR_KEY,
    image_size=IMAGE_SIZE, max_pairs_per_scene=100,
    min_overlap=0.20, augment=True)

val_ds = MegaDepth1500Dataset(
    SCENE_INFO_DIR, IMAGE_ROOT, IMG_KEY, PAIR_KEY,
    image_size=IMAGE_SIZE, max_pairs_per_scene=30,
    min_overlap=0.20, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, collate_fn=collate_fn)

print(f'Train: {len(train_ds):,} pairs  ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds):,} pairs  ({len(val_loader)} batches)')


In [ ]:
import torch.optim as optim
from torch.amp import GradScaler
from losses.hinge_loss import HomographyHingeLoss

# ── Loss ───────────────────────────────────────────────────────────────────
# lambda_rep > 0 enables the repeatability reward:
#   for positions with a ground-truth geometric match, high scores are rewarded.
#   Combined with the score-weighted hinge loss (which penalises scores at
#   non-repeatable positions), the kp_head receives a balanced gradient signal.
loss_fn = HomographyHingeLoss(
    positive_margin=1.0,
    negative_margin=0.2,
    lambda_d=250.0,
    lambda_rep=0.5,          # repeatability reward weight
    correspondence_threshold=8.0,
    safe_radius=8.0,
)

# ── SuperPoint is fully frozen — do NOT call requires_grad_(True) on it ───
# (the original notebook incorrectly unfroze SP params, which wasted optimizer
#  memory since SP is wrapped in torch.no_grad() and receives zero gradient)
for p in model.superpoint.parameters():
    p.requires_grad_(False)

trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
print(f'Optimising {n_trainable:,} parameters  (XFeat kp_head only)')

# ── Optimizer ─────────────────────────────────────────────────────────────
optimizer = optim.Adam(trainable_params, lr=1e-4, weight_decay=1e-4)

# ReduceLROnPlateau: reduce lr when val loss plateaus.
# More adaptive than fixed milestones — works well with early stopping.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

USE_AMP = False  # disable AMP until first epoch/backward is stable (no grid/grad RuntimeError)
scaler = GradScaler('cuda', enabled=USE_AMP)
print('✓ Loss / Optimizer / Scheduler configured')


In [ ]:
from collections import defaultdict
from pathlib import Path
from tqdm.notebook import tqdm
import torch, os, time, torch.nn as nn, glob

CKPT_DIR = DRIVE_CKPT_DIR
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Using persistent checkpoint directory: {CKPT_DIR}')

MAX_EPOCHS      = 30
GRAD_CLIP       = 10.0
SAVE_EVERY      = 5
PATIENCE        = 8     # early-stop: epochs without val improvement

history     = defaultdict(list)
best_val    = float('inf')
bad_epochs  = 0
start_epoch = 0

# Resume from latest Drive checkpoint when available
auto_resume = None
epoch_ckpts = sorted(glob.glob(f'{CKPT_DIR}/epoch_*.pth'))
if epoch_ckpts:
    auto_resume = epoch_ckpts[-1]
elif os.path.exists(f'{CKPT_DIR}/best.pth'):
    auto_resume = f'{CKPT_DIR}/best.pth'

if auto_resume is not None:
    print(f'Resuming from: {auto_resume}')
    state = torch.load(auto_resume, map_location=device)

    if 'model' not in state:
        raise KeyError(f"Checkpoint missing required 'model' key: {auto_resume}")

    load_result = model.load_state_dict(state['model'], strict=False)
    missing_keys = getattr(load_result, 'missing_keys', [])
    unexpected_keys = getattr(load_result, 'unexpected_keys', [])
    if missing_keys:
        print(f'⚠ Resume missing model keys: {len(missing_keys)} (showing first 10)')
        print(missing_keys[:10])
    if unexpected_keys:
        print(f'⚠ Resume unexpected model keys: {len(unexpected_keys)} (showing first 10)')
        print(unexpected_keys[:10])

    if 'optimizer' in state:
        optimizer.load_state_dict(state['optimizer'])
    if 'scaler' in state:
        scaler.load_state_dict(state['scaler'])

    if 'epoch' not in state:
        print('⚠ Resume checkpoint has no epoch key; defaulting start_epoch=0.')
        start_epoch = 0
    else:
        start_epoch = int(state['epoch']) + 1
    best_val = float(state.get('val_loss', best_val))

    loaded_history = state.get('history', {})
    if isinstance(loaded_history, dict):
        for k, v in loaded_history.items():
            history[k] = list(v)

    print(f'✓ Resume ready: start_epoch={start_epoch}, best_val={best_val:.4f}')
else:
    print('No checkpoint found in Drive folder. Starting from scratch.')


def run_epoch(loader, is_train, max_batches=None):
    """One epoch of training or validation."""
    model.train(is_train)
    agg   = defaultdict(float)
    n_upd = 0
    n_bat = 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, leave=False, desc='Train' if is_train else 'Val  ')
        for bi, batch in enumerate(pbar):
            if max_batches is not None and bi >= max_batches:
                break

            img1 = batch['image1'].to(device, non_blocking=True)
            img2 = batch['image2'].to(device, non_blocking=True)
            Hs   = batch['homography'].to(device, non_blocking=True)
            B    = img1.shape[0]

            # Optional depth-based warp field
            wf = batch.get('warp_field')
            wv = batch.get('warp_valid')
            wf = wf.to(device, non_blocking=True) if isinstance(wf, torch.Tensor) else None
            wv = wv.to(device, non_blocking=True) if isinstance(wv, torch.Tensor) else None

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type='cuda', enabled=(USE_AMP and torch.cuda.is_available())):
                out1 = model.forward_train(img1)
                out2 = model.forward_train(img2)

                # descriptors are now (N, 256) — no transpose needed ✓
                loss, stats = loss_fn.forward_batch(
                    desc1_list=out1['descriptors'],
                    desc2_list=out2['descriptors'],
                    kp1_list=out1['keypoints_px'],
                    kp2_list=out2['keypoints_px'],
                    homographies=Hs,
                    image2_hws=[(img2.shape[2], img2.shape[3])] * B,
                    scores1_list=out1.get('scores'),  # enables grad → kp_head ✓
                    scores2_list=out2.get('scores'),
                    warp_fields=wf,
                    warp_valids=wv,
                )

            if is_train:
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                n_upd += 1

            for k, v in stats.items():
                agg[k] += float(v)
            n_bat += 1

            avg_kp = sum(k.shape[0] for k in out1['keypoints_px']) / B
            pbar.set_postfix({
                'loss': f"{float(loss.detach()):.3f}",
                'hinge': f"{stats.get('hinge', 0):.3f}",
                'pos_sim': f"{stats.get('pos_sim_mean', 0):.2f}",
                'neg_sim': f"{stats.get('neg_sim_mean', 0):.2f}",
                'kp': f"{avg_kp:.0f}",
            })

    return {k: v / max(n_bat, 1) for k, v in agg.items()}, n_upd


print(f'Starting epoch {start_epoch} → {MAX_EPOCHS}')
print(f'Pairs: {len(train_ds):,} train / {len(val_ds):,} val')
print('=' * 65)

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time()

    tr_stats, n_steps = run_epoch(train_loader, is_train=True)
    va_stats, _       = run_epoch(val_loader,   is_train=False)

    if n_steps > 0:
        scheduler.step(va_stats.get('loss', float('inf')))

    elapsed  = time.time() - t0
    val_loss = va_stats.get('loss', float('inf'))
    lr       = optimizer.param_groups[0]['lr']
    is_best  = val_loss < best_val

    if is_best:
        best_val   = val_loss
        bad_epochs = 0
    else:
        bad_epochs += 1

    for k, v in tr_stats.items(): history[f'train_{k}'].append(v)
    for k, v in va_stats.items(): history[f'val_{k}'].append(v)
    history['epoch'].append(epoch)

    print(
        f'Ep {epoch:02d}/{MAX_EPOCHS-1}  '
        f'train={tr_stats.get("loss",0):.4f}  '
        f'val={val_loss:.4f}  '
        f'hinge={va_stats.get("hinge",0):.4f}  '
        f'pos_sim={va_stats.get("pos_sim_mean",0):.3f}  '
        f'neg_sim={va_stats.get("neg_sim_mean",0):.3f}  '
        f'bad={bad_epochs}/{PATIENCE}  '
        f'lr={lr:.1e}  {elapsed:.0f}s  '
        f'{"★ BEST" if is_best else ""}'
    )

    if (epoch+1) % SAVE_EVERY == 0 or is_best:
        ckpt = dict(epoch=epoch, model=model.state_dict(),
                    optimizer=optimizer.state_dict(),
                    scaler=scaler.state_dict(),
                    val_loss=val_loss, history=dict(history))
        path = f'{CKPT_DIR}/epoch_{epoch:04d}.pth'
        torch.save(ckpt, path)
        if is_best:
            torch.save(ckpt, f'{CKPT_DIR}/best.pth')
            print(f'  → Saved best (val={best_val:.4f})')

    if bad_epochs >= PATIENCE:
        print()
        print(f'Early stopping (no improvement for {PATIENCE} epochs).')
        break

print()
print(f'Training complete. Best val loss: {best_val:.4f}')






In [ ]:
import matplotlib.pyplot as plt

epochs = history.get('epoch', list(range(len(history.get('train_loss',[])))))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (t_key, v_key, title) in zip(axes, [
    ('train_loss',      'val_loss',      'Total Loss'),
    ('train_hinge',     'val_hinge',     'Hinge Loss'),
    ('train_pos_sim_mean', 'val_pos_sim_mean', 'Positive-Pair Cosine Sim'),
]):
    tv = history.get(t_key, [])
    vv = history.get(v_key, [])
    ep = list(range(len(tv)))
    if tv: ax.plot(ep, tv, label='train')
    if vv: ax.plot(ep[:len(vv)], vv, label='val')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_curves.png', dpi=120)
plt.show()
print('Saved training_curves.png')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

model.eval()

# Grab one validation pair
batch = next(iter(val_loader))
img1_t = batch['image1'][:1].to(device)
img2_t = batch['image2'][:1].to(device)

with torch.no_grad():
    out1 = model(img1_t)
    out2 = model(img2_t)

kp1  = out1['keypoints'][0].cpu().numpy()   # (N, 2) normalised
kp2  = out2['keypoints'][0].cpu().numpy()

H, W = img1_t.shape[2:]
# Denormalise
kp1_px = kp1 * np.array([[W-1, H-1]])
kp2_px = kp2 * np.array([[W-1, H-1]])

img1_np = img1_t[0, 0].cpu().numpy()
img2_np = img2_t[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img, kp, title in [
    (axes[0], img1_np, kp1_px, f'Image 1  ({len(kp1_px)} keypoints)'),
    (axes[1], img2_np, kp2_px, f'Image 2  ({len(kp2_px)} keypoints)'),
]:
    ax.imshow(img, cmap='gray')
    ax.scatter(kp[:, 0], kp[:, 1], s=6, c='lime', linewidths=0)
    ax.set_title(title); ax.axis('off')

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/keypoints_sample.png', dpi=120)
plt.show()
print(f'Img1 kp: {len(kp1_px)}   Img2 kp: {len(kp2_px)}')


## Fixed A/B Benchmark (Old vs New Checkpoint)

After training, run this section to compare two checkpoints on the **same held-out pairs** with LightGlue matching.
It reports:
- `inlier_ratio`
- `mma@1/3/5px`
- `precision`
- `n_matches`
- `sim_gap`, `repeatability_mean`

It also saves match visualizations with inliers/outliers.


In [ ]:
# Install LightGlue (clean reinstall) and verify import
import os, sys, glob, subprocess, importlib

def reinstall_and_verify_lightglue():
    # Clean reinstall to avoid partially installed wheel state in Colab
    # check=False is intentional: uninstall can fail harmlessly when package is absent.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'lightglue'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'lightglue'], check=True)
    importlib.invalidate_caches()
    return importlib.import_module('lightglue')

try:
    import lightglue
    print('✓ LightGlue already importable')
except Exception:
    print('LightGlue import failed. Running clean reinstall...')
    _ = reinstall_and_verify_lightglue()
    print('✓ LightGlue installed and import verified.')
    print('If this cell still raises ModuleNotFoundError/import errors in a fresh runtime, restart runtime once and re-run it.')

# Choose checkpoints to compare
# NEW = current run best; OLD = previous baseline if available
NEW_CKPT = f"{CKPT_DIR}/best.pth"
MANUAL_OLD_CKPT = None  # set this path if you have a historical baseline

assert os.path.exists(NEW_CKPT), f"Missing NEW_CKPT: {NEW_CKPT}"

# Auto-fallback for OLD_CKPT when no historical baseline is provided:
# pick an earlier epoch checkpoint from current run if available.
epoch_ckpts = sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pth"))
auto_old = epoch_ckpts[0] if epoch_ckpts else None

OLD_CKPT = MANUAL_OLD_CKPT or auto_old
if OLD_CKPT is not None and os.path.abspath(OLD_CKPT) == os.path.abspath(NEW_CKPT):
    print('⚠️ OLD_CKPT equals NEW_CKPT; switching to single-model validation mode.')
    OLD_CKPT = None
single_model_mode = OLD_CKPT is None

AB_OUT = f"{REPO_DIR}/benchmarks/ab_lightglue_colab"
SINGLE_OUT = f"{REPO_DIR}/benchmarks/single_model_lightglue_colab"

if single_model_mode:
    print('⚠️ No distinct OLD_CKPT found. Running single-model validation with NEW_CKPT as both old/new.')
    print('   Treat this output as your baseline for the next experiment.')
    os.makedirs(SINGLE_OUT, exist_ok=True)
    run_old = NEW_CKPT
    run_new = NEW_CKPT
    OUTPUT_DIR = SINGLE_OUT
else:
    assert os.path.exists(OLD_CKPT), f"Missing OLD_CKPT: {OLD_CKPT}"
    os.makedirs(AB_OUT, exist_ok=True)
    run_old = OLD_CKPT
    run_new = NEW_CKPT
    OUTPUT_DIR = AB_OUT
    print(f'OLD_CKPT: {run_old}')
    print(f'NEW_CKPT: {run_new}')

cmd = [
    'python', f'{REPO_DIR}/evaluate_ab_lightglue.py',
    '--mode', 'megadepth',
    '--data_root', '/content/megadepth/megadepth_test_1500',
    '--scene_info_dir', SCENE_INFO_DIR,
    '--old_ckpt', run_old,
    '--new_ckpt', run_new,
    '--num_pairs', '100',
    '--batch_size', '1',
    '--num_workers', '0',
    '--mma_thresholds', '1,3,5',
    '--precision_threshold', '3.0',
    '--save_vis_count', '12',
    '--output_dir', OUTPUT_DIR,
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print(f'✓ Benchmark complete: {OUTPUT_DIR}')
AB_OUT = OUTPUT_DIR  # downstream summary/visualization cells use this


In [ ]:
# Load benchmark summary and view key deltas
import yaml
from pprint import pprint

summary_path = f"{AB_OUT}/summary.yaml"
with open(summary_path, 'r') as f:
    summary = yaml.safe_load(f)

print('=== OLD ===')
pprint(summary['old'])
print('
=== NEW ===')
pprint(summary['new'])
print('
=== DELTA (new - old) ===')
pprint(summary['delta_new_minus_old'])


In [ ]:
# Show a few saved match visualizations (green=inliers, red=outliers)
import glob, matplotlib.pyplot as plt
from PIL import Image

old_imgs = sorted(glob.glob(f"{AB_OUT}/old_vis/*.png"))
new_imgs = sorted(glob.glob(f"{AB_OUT}/new_vis/*.png"))

n = min(3, len(old_imgs), len(new_imgs))
if n == 0:
    print('No visualization images found. Re-run benchmark cell above.')
else:
    fig, axes = plt.subplots(n, 2, figsize=(14, 4*n))
    if n == 1:
        axes = [axes]
    for i in range(n):
        axes[i][0].imshow(Image.open(old_imgs[i]))
        axes[i][0].set_title(f'OLD pair {i}')
        axes[i][0].axis('off')

        axes[i][1].imshow(Image.open(new_imgs[i]))
        axes[i][1].set_title(f'NEW pair {i}')
        axes[i][1].axis('off')
    plt.tight_layout()
